# `kick`: workbench

The definition lives in [`dataset.py`](dataset.py) and the evidence in [`report.md`](report.md). Edit `dataset.py`
and re-run a cell: `ds` picks up the change (the raw files are not read again unless `_load_raw` changed).

In [ ]:
from data_foundry.v2 import workbench

ds = workbench()
ds.df.head()

In [ ]:
ds.run_all_checks()["summary"]

In [ ]:
ds.preview_splits()

## Exploration

Migrated from the v1 notebook; adapt the cells where needed.

In [ ]:
# The analysis needs the original test set and an auction-location key, which the curated data drops.
import matplotlib.pyplot as plt
import pandas as pd

df = ds.df.copy()
test = pd.read_csv(ds.raw_dir / "test.csv")
df["Auction_VNZIP1"] = df["Auction"].astype(str) + df["VNZIP1"].astype(str)
test["Auction_VNZIP1"] = test["Auction"].astype(str) + test["VNZIP1"].astype(str)

In [ ]:
# Many auctions happen repeatedly
plt.hist(df["PurchDate"].groupby(df["Auction_VNZIP1"]).nunique().sort_values())
plt.xlabel("Number of unique purchase dates per auction")
plt.ylabel("Frequency")
plt.title("Distribution of unique purchase dates per auction")
plt.show()

In [ ]:
# Some auctions don't reappear over time, but most do.

tmp = df[["PurchDate", "Auction_VNZIP1"]].copy()
tmp["PurchDate"] = pd.to_datetime(tmp["PurchDate"])
tmp = tmp.dropna(subset=["PurchDate", "Auction_VNZIP1"])

# All observations as dots, with first/last colored differently
group_life = (
    tmp.groupby("Auction_VNZIP1")["PurchDate"]
       .agg(first_seen="min", last_seen="max", n_obs="size")
       .sort_values("first_seen")
)

# Optional: only show groups with enough rows
group_life_plot = group_life[group_life["n_obs"] >= 20].copy()
groups_to_plot = group_life_plot.index

# Keep only rows from plotted groups
plot_df = tmp[tmp["Auction_VNZIP1"].isin(groups_to_plot)].copy()

# Map each group to a y-position
y_map = {g: i for i, g in enumerate(groups_to_plot)}
plot_df["y"] = plot_df["Auction_VNZIP1"].map(y_map)

# Flag first/last rows within each group
plot_df["is_first"] = (
    plot_df["PurchDate"]
    == plot_df.groupby("Auction_VNZIP1")["PurchDate"].transform("min")
)
plot_df["is_last"] = (
    plot_df["PurchDate"]
    == plot_df.groupby("Auction_VNZIP1")["PurchDate"].transform("max")
)

# Middle points are everything that's neither first nor last
middle = plot_df[~plot_df["is_first"] & ~plot_df["is_last"]]
firsts = plot_df[plot_df["is_first"]]
lasts = plot_df[plot_df["is_last"]]

fig, ax = plt.subplots(figsize=(14, 10))

# All points
ax.scatter(
    middle["PurchDate"], middle["y"],
    s=8, alpha=0.5, label="other observations"
)

# First seen points
ax.scatter(
    firsts["PurchDate"], firsts["y"],
    s=30, label="first seen"
)

# Last seen points
ax.scatter(
    lasts["PurchDate"], lasts["y"],
    s=30, label="last seen"
)

ax.set_yticks(range(len(groups_to_plot)))
ax.set_yticklabels(groups_to_plot)
ax.set_title("All Auction_VNZIP1 observations over time")
ax.set_xlabel("PurchDate")
ax.set_ylabel("Auction_VNZIP1")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Proof that the original train/test split was not random, but based on "Auction" and "VNZIP1". 
intersection = set(df["Auction"].astype(str) + df["VNZIP1"].astype(str)).intersection(set(test["VNZIP1"].astype(str)))
union = set(df["Auction"].astype(str) + df["VNZIP1"].astype(str)).union(set(test["VNZIP1"].astype(str)))

len(intersection) / len(union)

## Report

`ds.check()` runs the full pipeline and the bundle checks without saving and writes `report.md`. `ds.build()` also
saves the container (new UUID); only the curator runs it.

In [ ]:
result = ds.check()